# 39 · Spelling correction, rewriting, and semantic drift

Units: S06. Prerequisites: 38.

Sources: [Wang Shusen’s pinned outline](https://github.com/wangshusen/SearchEngine/tree/7339c6b126e314abdb74a89adb033d39d8e3d37b); [Manning et al., Introduction to Information Retrieval](https://nlp.stanford.edu/IR-book/) (supplementary original textbook, chapters 1, 2, 6, and 8). This is a Lab-authored exercise. Wang's last four modules have a public outline only; this is not presented as the author's complete lecture material.

Evidence boundary: artificial spelling, synonym, and constraint examples, not generative-model effectiveness.

Learning method: predict each comparison, change one variable, run, and explain the difference.

> **English companion.** All instructional Markdown is translated. Experiment code, Chinese language examples, execution counts, and saved outputs remain as in the source edition; historical output text may therefore be Chinese. The initial setup cell anchors the working directory and Python import path for this companion's deeper folder; it does not change the experiment. Saved outputs are historical evidence, not a new run. Run from the top after preparing the required data and models.

In [ ]:
# English companions live one directory deeper; keep the original experiment code unchanged.
from pathlib import Path as _TranslationPath
import os as _translation_os
import sys as _translation_sys
_translation_root = next((p for p in [_TranslationPath.cwd(), *_TranslationPath.cwd().parents]
                          if (p / "reclab").is_dir()), None)
if _translation_root is None:
    raise RuntimeError("Open this notebook from inside the Search Recommendation Ads Lab checkout.")
_translation_os.chdir(_translation_root)
if str(_translation_root) not in _translation_sys.path:
    _translation_sys.path.insert(0, str(_translation_root))


In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
from reclab.learning.search_curriculum import *

## Conservative rules
Fix mappings battrey→battery and cellphone→phone. Preserve both original and rewritten queries. Entities, negation, and constraints must survive.

In [2]:
for q in ['battrey repair','cellphone repair official']:
    print(q,'→',rewrite(q))
assert 'official' in rewrite('cellphone repair official')

battrey repair → battery repair
cellphone repair official → phone repair official


## Compare with the original query
Artificial gold is a/c/d. Change only spelling, report missed and falsely retrieved sets, and do not directly compare scores across queries.

In [3]:
gold={'a','c','d'}
q='battrey'
for name,x in [('original',q),('corrected',rewrite(q))]:
    ids={d for d,s in field_rank(x)}
    print(name,sorted(ids),'miss',sorted(gold-ids),'false',sorted(ids-gold))

original [] miss ['a', 'c', 'd'] false []
corrected ['a', 'b', 'c', 'd'] miss [] false ['b']


## A failed rewrite
Deliberately remove official and add fruit to create drift. Change only aggressive. Inspect constraint preservation instead of linguistic fluency.

In [4]:
q='apple repair official'
good=rewrite(q); bad=rewrite(q,aggressive=True)
print(good,field_rank(good,required=['official']))
print(bad,field_rank(bad))
assert 'official' not in bad

apple repair official [('a', 5.0)]
apple repair fruit [('b', 6.0), ('d', 6.0), ('a', 4.0), ('c', 3.0)]


## Fallback
If constraint validation fails, return to the original query. A new rewrite may still yield no results. Online evaluation would also need frequency, human judgments, and latency.

In [5]:
def guarded(q):
    x=rewrite(q,aggressive=True)
    return q if ('official' in terms(q) and 'official' not in terms(x)) else x
assert guarded(q)==q
print('fallback',guarded(q))

fallback apple repair official


## Interpreting the saved run

battrey initially retrieves nothing. Correcting it to battery retrieves documents but also matches spam, so correction does not guarantee precision. The aggressive rewrite removes official and adds fruit; the constraint check falls back to the original. Inspect gains, false retrievals, and constraint preservation together.

After changing parameters, interpret the new outputs. This fixed commentary is not a new measurement.

## Review and exercise

Write down inputs, outputs, what stayed fixed, and conclusions this chapter cannot support. Predict and rerun one single-variable comparison above; retain failures. Return to the search learning path for the next semantic unit.